## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# Data Preprocessing

Prepare aligned MIND-small news and behavior tables for temporal recommendation experiments. Inputs are the official source news/behavior TSVs; outputs are the combined processed tables, split summary and source metadata. The raw files remain unchanged.

## Identity and partitioning

News is deduplicated by `news_id`. Behavior rows retain `source` and `orig_impression_id`; a stable chronological ordering assigns a globally unique `impression_id`. Empty history is represented by an empty string so its later length is zero.

With $d_i$ the calendar-day offset from the first observation,

$$\mathrm{split}(i)=\begin{cases}\mathrm{train}&d_i\in\{0,1,2,3,4\}\\mathrm{val}&d_i=5\\mathrm{test}&d_i=6.\end{cases}$$

All seven days must be present and every row must belong to a partition. Explicit checks require the last train timestamp to precede validation and validation to precede test. Partitioning is chronological, not a random row split. Source-file signatures make prepared artifacts traceable to the supplied files.

## Output contract

`behaviors` contains all retained impressions and their split labels; `news` is the common article catalog. `split_summary.csv` reports counts and observation ranges. Parquet is used when available and pickle otherwise. Negative sampling and model fitting do not occur in this stage.


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features


In [2]:
import json
import numpy as np
import pandas as pd
from utils.data_utils import source_directory, NEWS_COLUMNS, BEHAVIOR_COLUMNS

SEED = 42
TRAIN_DAYS, VAL_DAY, TEST_DAY = [0, 1, 2, 3, 4], 5, 6
SPLITS = ["train", "val", "test"]
PREPROCESSING_DIR = PATHS["processed"] / "preprocessing"
PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)


## 1. Optional dataset download

Use existing TSVs or download the original MIND archives from Hugging Face. Drive locations remain configurable.


In [3]:
DOWNLOAD_MIND = False
HF_TOKEN = os.environ.get("HF_TOKEN", "")

if DOWNLOAD_MIND:
    import zipfile
    from huggingface_hub import hf_hub_download

    PATHS["raw"].mkdir(parents=True, exist_ok=True)
    for name in ["MINDsmall_train", "MINDsmall_dev"]:
        zip_path = PATHS["raw"] / f"{name}.zip"
        if not zip_path.exists():
            zip_path = Path(
                hf_hub_download(
                    "yjw1029/MIND",
                    f"{name}.zip",
                    repo_type="dataset",
                    token=HF_TOKEN or None,
                    local_dir=str(PATHS["raw"]),
                )
            )
        with zipfile.ZipFile(zip_path) as archive:
            destination = (
                PATHS["raw"]
                if all(p.startswith(name + "/") for p in archive.namelist())
                else PATHS["raw"] / name
            )
            archive.extractall(destination)


## 2. Source tables


In [4]:
beh_parts, news_parts = [], []

for source in ["train", "dev"]:
    directory = source_directory(PATHS["raw"], source)
    part = pd.read_csv(
        directory / "behaviors.tsv", sep="\t", header=None, names=BEHAVIOR_COLUMNS
    )
    part["source"] = source
    beh_parts.append(part)
    news_parts.append(
        pd.read_csv(
            directory / "news.tsv", sep="\t", header=None, names=NEWS_COLUMNS, quoting=3
        )
    )

beh = pd.concat(beh_parts, ignore_index=True)
news = (
    pd.concat(news_parts, ignore_index=True)
    .drop_duplicates("news_id")
    .reset_index(drop=True)
)
beh["time"] = pd.to_datetime(beh["time"], format="%m/%d/%Y %I:%M:%S %p")
beh = beh.sort_values(
    ["time", "source", "orig_impression_id"], kind="mergesort"
).reset_index(drop=True)
beh["impression_id"] = np.arange(1, len(beh) + 1)
beh["history"] = beh["history"].fillna("")
assert not beh.duplicated(["source", "orig_impression_id"]).any()
assert beh["impression_id"].is_unique and news["news_id"].is_unique


## 3. Temporal partitions


In [5]:
first_day = beh["time"].dt.normalize().min()
beh["day"] = (beh["time"].dt.normalize() - first_day).dt.days.astype(int)
n_days = beh["day"].nunique()
assert (
    n_days == 7
), f"Expected 7 days of logs, found {n_days}: {sorted(beh.day.unique())}"
beh["split"] = np.select(
    [beh.day.isin(TRAIN_DAYS), beh.day.eq(VAL_DAY), beh.day.eq(TEST_DAY)],
    ["train", "val", "test"],
    default="drop",
)
assert (beh.split != "drop").all()
split_summary = (
    beh.groupby("split")
    .agg(
        impressions=("impression_id", "size"),
        users=("user_id", "nunique"),
        start=("time", "min"),
        end=("time", "max"),
        days=("day", lambda s: str([int(x) for x in sorted(s.unique())])),
    )
    .reindex(SPLITS)
)
split_summary["share"] = (split_summary.impressions / len(beh)).round(3)
print(split_summary)
assert (
    beh.loc[beh.split == "train", "time"].max()
    < beh.loc[beh.split == "val", "time"].min()
)
assert (
    beh.loc[beh.split == "val", "time"].max()
    < beh.loc[beh.split == "test", "time"].min()
)


       impressions  users  ...             days  share
split                      ...                        
train       126695  46012  ...  [0, 1, 2, 3, 4]  0.551
val          30270  20179  ...              [5]  0.132
test         73152  50000  ...              [6]  0.318

[3 rows x 6 columns]


## 4. Export


In [6]:
save_table(beh, PREPROCESSING_DIR / "behaviors")
save_table(news, PREPROCESSING_DIR / "news")
split_summary.to_csv(PREPROCESSING_DIR / "split_summary.csv")
metadata = {
    "raw_directory": str(PATHS["raw"].resolve()),
    "first_day": str(first_day.date()),
    "train_days": TRAIN_DAYS,
    "val_day": VAL_DAY,
    "test_day": TEST_DAY,
    "impressions": len(beh),
    "articles": len(news),
}
metadata["source_files"] = source_fingerprint(PATHS)
(PREPROCESSING_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))


1316